<a href="https://colab.research.google.com/github/hinamehboobcs/Industry-Project/blob/main/DDQN%2BActionMasking%2BrollingHorizon%2BScenarioGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

# folders
os.makedirs("project/data", exist_ok=True)
os.makedirs("project/processed_data", exist_ok=True)
os.makedirs("project/results", exist_ok=True)

# python files
files = [
    "config.py",
    "preprocessing.py",
    "distance_utils.py",
    "rolling_horizon.py",
    "scenario_generator.py",
    "constraints.py",
    "action_mask.py",
    "environment.py",
    "ddqn_network.py",
    "replay_buffer.py",
    "ddqn_agent.py",
    "trainer.py",
    "scheduler.py",
    "evaluation.py",
    "main.py"
]

for file in files:
    open(f"project/{file}", "w").close()

print("Project structure created successfully.")

Project structure created successfully.


In [3]:
import sys

sys.path.append("project")

from preprocessing import DataPreprocessor

processor = DataPreprocessor()

carers, visits = processor.run()


DATASET SHAPES
Carers : (462, 23)
Visits : (3118, 15)

CARERS MISSING VALUES
CarerID                   0
WorkerType                0
Latitude                  0
Longitude                 0
PreferredAreaLatitude     0
PreferredAreaLongitude    0
MaxTravelDistanceMiles    0
ExperienceInYears         0
Skills                    0
Gender                    0
Languages                 0
VehicleType               0
WorkingDays               0
OffDays                   0
ShiftStart                0
ShiftEnd                  0
ContractHoursPerWeek      0
MaxDailyHours             0
MaxWeeklyHours            0
PreferredShift            0
Date                      0
DayOfWeek                 0
Status                    0
dtype: int64

VISITS MISSING VALUES
VisitID                  0
PatientID                0
Day                      0
Latitude                 0
Longitude                0
VisitDurationMinutes     0
TimeWindowStart          0
TimeWindowEnd            0
PriorityLevel            0

In [4]:
import sys
sys.path.append("project")

from preprocessing import DataPreprocessor

processor = DataPreprocessor()

carers, visits = processor.run()


DATASET SHAPES
Carers : (462, 23)
Visits : (3118, 15)

CARERS MISSING VALUES
CarerID                   0
WorkerType                0
Latitude                  0
Longitude                 0
PreferredAreaLatitude     0
PreferredAreaLongitude    0
MaxTravelDistanceMiles    0
ExperienceInYears         0
Skills                    0
Gender                    0
Languages                 0
VehicleType               0
WorkingDays               0
OffDays                   0
ShiftStart                0
ShiftEnd                  0
ContractHoursPerWeek      0
MaxDailyHours             0
MaxWeeklyHours            0
PreferredShift            0
Date                      0
DayOfWeek                 0
Status                    0
dtype: int64

VISITS MISSING VALUES
VisitID                  0
PatientID                0
Day                      0
Latitude                 0
Longitude                0
VisitDurationMinutes     0
TimeWindowStart          0
TimeWindowEnd            0
PriorityLevel            0

In [5]:
#Distance Engine
from distance_utils import DistanceEngine

distance_engine = DistanceEngine(
    carers,
    visits
)

distance_matrix = \
    distance_engine.build_distance_matrix()


Building distance matrix...
Distance matrix shape: (3118, 462)

Distance matrix saved.
Rows: 1440516


In [7]:
%%writefile project/distance_lookup.py

Writing project/distance_lookup.py


In [12]:
from distance_lookup import DistanceLookup
print("DistanceLookup imported successfully")

DistanceLookup imported successfully


In [13]:
#Sanity Check of Environement
from environment import CareSchedulingEnv
from preprocessing import DataPreprocessor
from distance_utils import DistanceEngine

# load data
processor = DataPreprocessor()
carers, visits = processor.run()

distance_engine = DistanceEngine(carers, visits)
distance_matrix = distance_engine.build_distance_matrix()

# init environment
env = CareSchedulingEnv(carers, visits, distance_matrix)

# reset for one day
state = env.reset("Monday")

print("State loaded:", type(state))
print("Active visits:", len(state["active_visits"]))


DATASET SHAPES
Carers : (462, 23)
Visits : (3118, 15)

CARERS MISSING VALUES
CarerID                   0
WorkerType                0
Latitude                  0
Longitude                 0
PreferredAreaLatitude     0
PreferredAreaLongitude    0
MaxTravelDistanceMiles    0
ExperienceInYears         0
Skills                    0
Gender                    0
Languages                 0
VehicleType               0
WorkingDays               0
OffDays                   0
ShiftStart                0
ShiftEnd                  0
ContractHoursPerWeek      0
MaxDailyHours             0
MaxWeeklyHours            0
PreferredShift            0
Date                      0
DayOfWeek                 0
Status                    0
dtype: int64

VISITS MISSING VALUES
VisitID                  0
PatientID                0
Day                      0
Latitude                 0
Longitude                0
VisitDurationMinutes     0
TimeWindowStart          0
TimeWindowEnd            0
PriorityLevel            0

KeyError: 'active_visits'

In [14]:
from environment import CareSchedulingEnv
from preprocessing import DataPreprocessor
from distance_utils import DistanceEngine

processor = DataPreprocessor()
carers, visits = processor.run()

distance_engine = DistanceEngine(carers, visits)
distance_matrix = distance_engine.build_distance_matrix()

env = CareSchedulingEnv(carers, visits, distance_matrix)

state = env.reset("Monday")

print("State type:", type(state))
print("Shape:", state.shape)
print("Columns:", state.columns[:5])


DATASET SHAPES
Carers : (462, 23)
Visits : (3118, 15)

CARERS MISSING VALUES
CarerID                   0
WorkerType                0
Latitude                  0
Longitude                 0
PreferredAreaLatitude     0
PreferredAreaLongitude    0
MaxTravelDistanceMiles    0
ExperienceInYears         0
Skills                    0
Gender                    0
Languages                 0
VehicleType               0
WorkingDays               0
OffDays                   0
ShiftStart                0
ShiftEnd                  0
ContractHoursPerWeek      0
MaxDailyHours             0
MaxWeeklyHours            0
PreferredShift            0
Date                      0
DayOfWeek                 0
Status                    0
dtype: int64

VISITS MISSING VALUES
VisitID                  0
PatientID                0
Day                      0
Latitude                 0
Longitude                0
VisitDurationMinutes     0
TimeWindowStart          0
TimeWindowEnd            0
PriorityLevel            0

In [17]:
print("Carers in env:", len(carers))
print("Mask size should be:", len(carers))

Carers in env: 462
Mask size should be: 462


In [18]:
from action_mask import ActionMasker

masker = ActionMasker(carers, visits, distance_matrix)

visit_id = visits.iloc[0]["VisitID"]

mask = masker.get_mask(
    visit_id,
    {c: 0 for c in carers["CarerID"]}
)

print("Valid actions:", mask.sum())
print("Mask shape:", mask.shape)

IndexError: index 101 is out of bounds for axis 0 with size 100

In [19]:
print("Carers from preprocessing:", carers.shape)

from environment import CareSchedulingEnv
env = CareSchedulingEnv(carers, visits, distance_matrix)

print("Carers in environment:", env.carers.shape)

from action_mask import ActionMasker
masker = ActionMasker(carers, visits, distance_matrix)

print("Carers in masker:", masker.carers.shape)
print("Mask size:", len(masker.carer_ids))
print("Index max:", max(masker.carer_index.values()))

Carers from preprocessing: (462, 23)
Carers in environment: (462, 23)
Carers in masker: (462, 23)


AttributeError: 'ActionMasker' object has no attribute 'carer_ids'

In [20]:
import importlib
import action_mask

importlib.reload(action_mask)

from action_mask import ActionMasker

print("Reloaded ActionMasker successfully")

Reloaded ActionMasker successfully


In [21]:
import sys

sys.modules.pop("action_mask", None)

<module 'action_mask' from '/content/project/action_mask.py'>

In [22]:
from action_mask import ActionMasker

In [23]:
masker = ActionMasker(carers, visits, distance_matrix)

visit_id = visits.iloc[0]["VisitID"]

mask = masker.get_mask(
    visit_id,
    {c: 0 for c in carers["CarerID"]}
)

print("Valid actions:", mask.sum())
print("Mask shape:", mask.shape)

Valid actions: 362.0
Mask shape: (462,)


In [25]:
import torch
import importlib

from preprocessing import DataPreprocessor
from distance_utils import DistanceEngine
from trainer import Trainer

# reload modules (important in Colab)
import preprocessing
import distance_utils
import trainer

importlib.reload(preprocessing)
importlib.reload(distance_utils)
importlib.reload(trainer)

from preprocessing import DataPreprocessor
from distance_utils import DistanceEngine
from trainer import Trainer


print("Loading data...")

processor = DataPreprocessor()
carers, visits = processor.run()

print("Building distance matrix...")

distance_engine = DistanceEngine(carers, visits)
distance_matrix = distance_engine.build_distance_matrix()

print("Initializing trainer...")

state_size = 7
action_size = len(carers)

device = "cuda" if torch.cuda.is_available() else "cpu"

trainer = Trainer(
    carers=carers,
    visits=visits,
    distance_matrix=distance_matrix,
    state_size=state_size,
    action_size=action_size,
    device=device
)

print("Trainer ready ✔")

Loading data...

DATASET SHAPES
Carers : (462, 23)
Visits : (3118, 15)

CARERS MISSING VALUES
CarerID                   0
WorkerType                0
Latitude                  0
Longitude                 0
PreferredAreaLatitude     0
PreferredAreaLongitude    0
MaxTravelDistanceMiles    0
ExperienceInYears         0
Skills                    0
Gender                    0
Languages                 0
VehicleType               0
WorkingDays               0
OffDays                   0
ShiftStart                0
ShiftEnd                  0
ContractHoursPerWeek      0
MaxDailyHours             0
MaxWeeklyHours            0
PreferredShift            0
Date                      0
DayOfWeek                 0
Status                    0
dtype: int64

VISITS MISSING VALUES
VisitID                  0
PatientID                0
Day                      0
Latitude                 0
Longitude                0
VisitDurationMinutes     0
TimeWindowStart          0
TimeWindowEnd            0
PriorityLe

In [29]:
import sys
import importlib

if "trainer" in sys.modules:
    del sys.modules["trainer"]

import trainer
importlib.reload(trainer)

print("Trainer module reloaded cleanly ✔")

Trainer module reloaded cleanly ✔


In [30]:
from preprocessing import DataPreprocessor
from distance_utils import DistanceEngine
from trainer import Trainer
import torch

processor = DataPreprocessor()
carers, visits = processor.run()

distance_engine = DistanceEngine(carers, visits)
distance_matrix = distance_engine.build_distance_matrix()

state_size = 7
action_size = len(carers)

device = "cuda" if torch.cuda.is_available() else "cpu"

trainer = Trainer(
    carers=carers,
    visits=visits,
    distance_matrix=distance_matrix,
    state_size=state_size,
    action_size=action_size,
    device=device
)

print("Trainer rebuilt ✔")


DATASET SHAPES
Carers : (462, 23)
Visits : (3118, 15)

CARERS MISSING VALUES
CarerID                   0
WorkerType                0
Latitude                  0
Longitude                 0
PreferredAreaLatitude     0
PreferredAreaLongitude    0
MaxTravelDistanceMiles    0
ExperienceInYears         0
Skills                    0
Gender                    0
Languages                 0
VehicleType               0
WorkingDays               0
OffDays                   0
ShiftStart                0
ShiftEnd                  0
ContractHoursPerWeek      0
MaxDailyHours             0
MaxWeeklyHours            0
PreferredShift            0
Date                      0
DayOfWeek                 0
Status                    0
dtype: int64

VISITS MISSING VALUES
VisitID                  0
PatientID                0
Day                      0
Latitude                 0
Longitude                0
VisitDurationMinutes     0
TimeWindowStart          0
TimeWindowEnd            0
PriorityLevel            0

In [31]:
trainer.train(episodes=1, batch_size=32)


EPISODE (WEEK): 0

[RESET] Day initialized: Monday
Active visits: 570


IndexError: index 101 is out of bounds for axis 0 with size 100

In [32]:
print("Env carers:", len(trainer.env.carers))
print("Masker carers:", len(trainer.env.masker.carers))
print("Controller workload keys:", len(trainer.env.controller.carer_workload))

Env carers: 462
Masker carers: 462
Controller workload keys: 100


In [33]:
print("Carers:", len(carers))
print("Workload keys:", len(trainer.env.controller.carer_workload))
print("Masker carers:", len(trainer.env.masker.carers))

Carers: 462
Workload keys: 100
Masker carers: 462


In [34]:
print("Carers:", len(carers))
print("Workload keys:", len(trainer.env.controller.carer_workload))
print("Masker carers:", len(trainer.env.masker.carers))

Carers: 462
Workload keys: 100
Masker carers: 462


In [35]:
import inspect
from environment import CareSchedulingEnv

print(inspect.getsource(CareSchedulingEnv))

class CareSchedulingEnv:

    def __init__(self,
                 carers,
                 visits,
                 distance_matrix):

        # ----------------------------
        # FORCE GLOBAL CONSISTENCY
        # ----------------------------
        self.carers = carers.copy().reset_index(drop=True)
        self.visits = visits.copy().reset_index(drop=True)

        self.distance_matrix = distance_matrix

        # ----------------------------
        # CONTROLLER (NO FILTERING)
        # ----------------------------
        self.controller = RollingHorizonController(
            self.carers,
            self.visits,
            self.distance_matrix
        )

        # ----------------------------
        # ACTION MASK (FULL SPACE)
        # ----------------------------
        self.masker = ActionMasker(
            self.carers,
            self.visits,
            self.distance_matrix
        )

        self.current_day = None

    #############################################

In [36]:
from rolling_horizon import RollingHorizonController
import inspect

print(inspect.getsource(RollingHorizonController))

class RollingHorizonController:

    def __init__(self,
                 carers,
                 visits,
                 distance_matrix):

        # ----------------------------
        # FIX: NEVER reduce carers
        # ----------------------------
        self.carers = carers.copy().reset_index(drop=True)
        self.visits = visits.copy().reset_index(drop=True)

        self.distance_matrix = distance_matrix

        # ----------------------------
        # GLOBAL WORKLOAD (FIXED SIZE = ALL CARERS)
        # ----------------------------
        self.carer_workload = {
            cid: 0 for cid in self.carers["CarerID"].tolist()
        }

        # current day state
        self.current_day = None
        self.active_visits = None

    ########################################################
    # RESET DAY (ROLLING HORIZON ENTRY POINT)
    ########################################################

    def reset_day(self, day):

        self.current_day = day

        self.act

In [37]:
inspect.getsource(CareSchedulingEnv)
inspect.getsource(RollingHorizonController)

'class RollingHorizonController:\n\n    def __init__(self,\n                 carers,\n                 visits,\n                 distance_matrix):\n\n        # ----------------------------\n        # FIX: NEVER reduce carers\n        # ----------------------------\n        self.carers = carers.copy().reset_index(drop=True)\n        self.visits = visits.copy().reset_index(drop=True)\n\n        self.distance_matrix = distance_matrix\n\n        # ----------------------------\n        # GLOBAL WORKLOAD (FIXED SIZE = ALL CARERS)\n        # ----------------------------\n        self.carer_workload = {\n            cid: 0 for cid in self.carers["CarerID"].tolist()\n        }\n\n        # current day state\n        self.current_day = None\n        self.active_visits = None\n\n    ########################################################\n    # RESET DAY (ROLLING HORIZON ENTRY POINT)\n    ########################################################\n\n    def reset_day(self, day):\n\n        self.cu

In [38]:
import sys

# remove cached modules
for m in list(sys.modules.keys()):
    if "environment" in m or "rolling" in m or "action" in m or "trainer" in m:
        del sys.modules[m]

print("Modules cleared ✔")

Modules cleared ✔


In [39]:
from preprocessing import DataPreprocessor
from distance_utils import DistanceEngine

from environment import CareSchedulingEnv
from trainer import Trainer

import torch

In [40]:
processor = DataPreprocessor()
carers, visits = processor.run()

distance_engine = DistanceEngine(carers, visits)
distance_matrix = distance_engine.build_distance_matrix()

trainer = Trainer(
    carers=carers,
    visits=visits,
    distance_matrix=distance_matrix,
    state_size=7,
    action_size=len(carers),
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Trainer rebuilt ✔")


DATASET SHAPES
Carers : (462, 23)
Visits : (3118, 15)

CARERS MISSING VALUES
CarerID                   0
WorkerType                0
Latitude                  0
Longitude                 0
PreferredAreaLatitude     0
PreferredAreaLongitude    0
MaxTravelDistanceMiles    0
ExperienceInYears         0
Skills                    0
Gender                    0
Languages                 0
VehicleType               0
WorkingDays               0
OffDays                   0
ShiftStart                0
ShiftEnd                  0
ContractHoursPerWeek      0
MaxDailyHours             0
MaxWeeklyHours            0
PreferredShift            0
Date                      0
DayOfWeek                 0
Status                    0
dtype: int64

VISITS MISSING VALUES
VisitID                  0
PatientID                0
Day                      0
Latitude                 0
Longitude                0
VisitDurationMinutes     0
TimeWindowStart          0
TimeWindowEnd            0
PriorityLevel            0

In [41]:
print("Carers:", len(carers))
print("Env carers:", len(trainer.env.carers))
print("Masker carers:", len(trainer.env.masker.carers))
print("Workload keys:", len(trainer.env.controller.carer_workload))

Carers: 462
Env carers: 462
Masker carers: 462
Workload keys: 100


In [42]:
print("Workload keys:", len(trainer.env.controller.carer_workload))

Workload keys: 100


In [43]:
env = trainer.env

print("ENV OBJECT:", env)
print("CONTROLLER OBJECT:", env.controller)

print("\n--- CARER WORKLOAD TYPE ---")
print(type(env.controller.carer_workload))

print("\n--- SAMPLE KEYS ---")
keys = list(env.controller.carer_workload.keys())
print("First 10 keys:", keys[:10])
print("Total keys:", len(keys))

print("\n--- CARERS IN SYSTEM ---")
print("env.carers:", len(env.carers))
print("controller.carers:", len(env.controller.carers))

ENV OBJECT: <environment.CareSchedulingEnv object at 0x7bb31d0ee210>
CONTROLLER OBJECT: <rolling_horizon.RollingHorizonController object at 0x7bb32cbde7b0>

--- CARER WORKLOAD TYPE ---
<class 'dict'>

--- SAMPLE KEYS ---
First 10 keys: ['C0001', 'C0002', 'C0003', 'C0004', 'C0005', 'C0006', 'C0007', 'C0008', 'C0009', 'C0010']
Total keys: 100

--- CARERS IN SYSTEM ---
env.carers: 462
controller.carers: 462


In [46]:
from preprocessing import DataPreprocessor
from distance_utils import DistanceEngine
from trainer import Trainer
import torch

# ---------------------------
# LOAD DATA
# ---------------------------
processor = DataPreprocessor()
carers, visits = processor.run()

# ---------------------------
# BUILD DISTANCE MATRIX
# ---------------------------
engine = DistanceEngine(carers, visits)
distance_matrix = engine.build_distance_matrix()

# ---------------------------
# DEFINE RL DIMENSIONS
# ---------------------------
state_size = 7
action_size = len(carers)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------------------
# CREATE TRAINER (CORRECT WAY)
# ---------------------------
trainer = Trainer(
    carers,
    visits,
    distance_matrix,
    state_size,
    action_size,
    device
)

print("Trainer created successfully ✔")


DATASET SHAPES
Carers : (462, 23)
Visits : (3118, 15)

CARERS MISSING VALUES
CarerID                   0
WorkerType                0
Latitude                  0
Longitude                 0
PreferredAreaLatitude     0
PreferredAreaLongitude    0
MaxTravelDistanceMiles    0
ExperienceInYears         0
Skills                    0
Gender                    0
Languages                 0
VehicleType               0
WorkingDays               0
OffDays                   0
ShiftStart                0
ShiftEnd                  0
ContractHoursPerWeek      0
MaxDailyHours             0
MaxWeeklyHours            0
PreferredShift            0
Date                      0
DayOfWeek                 0
Status                    0
dtype: int64

VISITS MISSING VALUES
VisitID                  0
PatientID                0
Day                      0
Latitude                 0
Longitude                0
VisitDurationMinutes     0
TimeWindowStart          0
TimeWindowEnd            0
PriorityLevel            0

In [47]:
print("Carers:", len(carers))
print("Env carers:", len(trainer.env.carers))
print("Controller carers:", len(trainer.env.controller.carers))
print("Workload keys:", len(trainer.env.controller.carer_workload))

Carers: 462
Env carers: 462
Controller carers: 462
Workload keys: 100


In [48]:
print("Mask size:", len(trainer.env.masker.carer_ids))
print("Mask shape:", trainer.env.action_mask(visits.iloc[0]['VisitID']).shape)

Mask size: 462
Mask shape: (462,)


In [49]:
print("Action space size:", len(trainer.env.carers))
print("Mask space size:", len(trainer.env.action_mask(visits.iloc[0]['VisitID'])))

Action space size: 462
Mask space size: 462


In [54]:
trainer.train(episodes=1, batch_size=32)


EPISODE (WEEK): 0
[RESET] Day initialized: Monday
Active visits: 570


IndexError: single positional indexer is out-of-bounds